In [1]:
import os
import json
os.makedirs("data/json_files", exist_ok=True)

In [2]:
json_data = {
    "company": "TechCorp",
    "employees": [
        {
            "id": 1,
            "name": "John Doe",
            "role": "Software Engineer",
            "skills": ["Python", "JavaScript", "React"],
            "projects": [
                {"name": "RAG System", "status": "In Progress"},
                {"name": "Data Pipeline", "status": "Completed"},
            ],
        },
        {
            "id": 2,
            "name": "Jane Smith",
            "role": "Data Scientist",
            "skills": ["Python", "Machine Learning", "SQL"],
            "projects": [
                {"name": "ML Model", "status": "In Progress"},
                {"name": "Analytics Dashboard", "status": "Planning"},
            ],
        },
    ],
    "departments": {
        "engineering": {
            "head": "Mike Johnson",
            "budget": 1000000,
            "team_size": 25,
        },
        "data_science": {
            "head": "Sarah Williams",
            "budget": 750000,
            "team_size": 15,
        },
    },
}

In [4]:
with open("data/json_files/company.json", "w", encoding="utf-8") as f:
    json.dump(json_data, f, indent=4)

In [5]:
#Save JSON Lines format
jsonl_data = [
    {"timestamp": "2024-01-01T12:00:00Z", "event": "user_signup", "user_id": 123},
    {"timestamp": "2024-01-01T12:05:00Z", "event": "page_view", "user_id": 123, "page": "/home"},
    {"timestamp": "2024-01-01T12:10:00Z", "event": "purchase", "user_id": 123, "amount": 99.99}
]
with open("data/json_files/events.jsonl", "w", encoding="utf-8") as f:
    for record in jsonl_data:
        f.write(json.dumps(record) + "\n")

In [9]:
## Json Processing Strategies

In [10]:
from langchain_community.document_loaders import JSONLoader
import json

# Method 1: Json loader with jq_schema
json_loader =JSONLoader(
    file_path="data/json_files/company.json",
    jq_schema=".departments[]", # jq query to extract employee records
    text_content=False # Get full JSON objects instead of text
)
json_docs = json_loader.load()
print(f"Loaded {len(json_docs)} employee records")
json_docs[0].page_content # Full JSON for first employee


Loaded 2 employee records


'{"head": "Mike Johnson", "budget": 1000000, "team_size": 25}'

In [9]:
print(json_docs)

[Document(metadata={'source': '/home/divyanshu/Desktop/RAG/1-DataIngestParsing/data/json_files/company.json', 'seq_num': 1}, page_content='{"id": 1, "name": "John Doe", "role": "Software Engineer", "skills": ["Python", "JavaScript", "React"], "projects": [{"name": "RAG System", "status": "In Progress"}, {"name": "Data Pipeline", "status": "Completed"}]}'), Document(metadata={'source': '/home/divyanshu/Desktop/RAG/1-DataIngestParsing/data/json_files/company.json', 'seq_num': 2}, page_content='{"id": 2, "name": "Jane Smith", "role": "Data Scientist", "skills": ["Python", "Machine Learning", "SQL"], "projects": [{"name": "ML Model", "status": "In Progress"}, {"name": "Analytics Dashboard", "status": "Planning"}]}')]


In [ ]:
#Method 2: Custome Json parsing for COnmplex Structures
from typing import List
from langchain_core.documents import Document
def process_json_intelligentlt(filepath1: str, filepath2: str) -> List[Document]:
    """" Custom JSON processing to handle complex nested structures intelligently."""

    # Load the JSON file
    with open(filepath1,"r", encoding="utf-8") as f:
        data = json.load(f)
    with open(filepath2,"r", encoding="utf-8") as f:
        data2 = [json.loads(line) for line in f if line.strip()] # parse line-by-line with json.loads() to handle JSON Lines format, if empty lines are present, they will be skipped
    # Store documents
    documents = []

    # Strategy1: Create documets for each employee with key details
    for emp in data.get("employees", []):
        content = f"""Employee Details:
        Name: {emp["name"]}
        Role: {emp["role"]}
        Skills: {', '.join(emp["skills"])}
        Projects:"""
        for proj in emp.get("projects", []):
            content += f"\n- {proj['name']} (Status: {proj['status']})"


    for event in data2:
        m1 = event["timestamp"]
        m2 = event["event"]
        m3 = event["user_id"]

        # Document 
        doc = Document(
            page_content = content,
            metadata = {
                "source": filepath1,
                "employee_name": emp["name"],
                "employee_id": emp["id"],
                "role": emp["role"],
                "timeStamp": m1,
                "event": m2,
                "user_id": m3,
                "data_type": "employee_record"        
            }
        )
        documents.append(doc)
    return documents

In [18]:
json_docs = process_json_intelligentlt("data/json_files/company.json", "data/json_files/events.jsonl")
print(f"Processed {len(json_docs)} employee documents") 
print(json_docs[0].page_content)
print(json_docs[0].metadata)

Processed 3 employee documents
Employee Details:
        Name: Jane Smith
        Role: Data Scientist
        Skills: Python, Machine Learning, SQL
        Projects:
- ML Model (Status: In Progress)
- Analytics Dashboard (Status: Planning)
{'source': 'data/json_files/company.json', 'employee_name': 'Jane Smith', 'employee_id': 2, 'role': 'Data Scientist', 'timeStamp': '2024-01-01T12:00:00Z', 'event': 'user_signup', 'user_id': 123, 'data_type': 'employee_record'}


In [19]:
print(json_docs)

[Document(metadata={'source': 'data/json_files/company.json', 'employee_name': 'Jane Smith', 'employee_id': 2, 'role': 'Data Scientist', 'timeStamp': '2024-01-01T12:00:00Z', 'event': 'user_signup', 'user_id': 123, 'data_type': 'employee_record'}, page_content='Employee Details:\n        Name: Jane Smith\n        Role: Data Scientist\n        Skills: Python, Machine Learning, SQL\n        Projects:\n- ML Model (Status: In Progress)\n- Analytics Dashboard (Status: Planning)'), Document(metadata={'source': 'data/json_files/company.json', 'employee_name': 'Jane Smith', 'employee_id': 2, 'role': 'Data Scientist', 'timeStamp': '2024-01-01T12:05:00Z', 'event': 'page_view', 'user_id': 123, 'data_type': 'employee_record'}, page_content='Employee Details:\n        Name: Jane Smith\n        Role: Data Scientist\n        Skills: Python, Machine Learning, SQL\n        Projects:\n- ML Model (Status: In Progress)\n- Analytics Dashboard (Status: Planning)'), Document(metadata={'source': 'data/json_fil